# Etapa 1 — Oracle de Koopman para el ciclo de cuatro fases

Antes de introducir un encoder neuronal, validamos toda la matemática con las features verdaderas. La fase evoluciona como `r → r+1 mod 4` y usamos las indicadoras centradas `ψ_c(r)=e_r−¼1`. Su span tiene dimensión 3 y el espectro esperado es `{-1, i, -i}`.

La matriz completa ajustada no está identificada sobre la dirección constante eliminada. Por eso el gate se calcula únicamente sobre el subespacio activo: rango 3 y errores one-step, invariancia, entrelazamiento, espectro, left eigenvectors y rollouts de ocho pasos menores que `1e-10`.

In [ ]:
# ruff: noqa: E402, E501
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from koopman_jepa.koopman import (
    centered_phase_indicators,
    cyclic_phase_operator,
    fit_linear_operator,
    left_eigendecomposition,
    linear_rollout,
    match_eigenvalues,
    restrict_operator,
    sample_span_basis,
)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
NUM_PHASES = 4
REPEATS = 64
TOLERANCE = 1e-10
phases = np.tile(np.arange(NUM_PHASES), REPEATS)
next_phases = (phases + 1) % NUM_PHASES
current = centered_phase_indicators(phases, NUM_PHASES)
future = centered_phase_indicators(next_phases, NUM_PHASES)
true_operator = cyclic_phase_operator(NUM_PHASES)
fitted_operator = fit_linear_operator(current, future)
basis = sample_span_basis(current)
fitted_active = restrict_operator(fitted_operator, basis)
true_active = restrict_operator(true_operator, basis)
assert basis.shape == (4, 3)
print("Fitted ambient operator:")
print(np.array2string(fitted_operator, precision=3, suppress_small=True))
print("Active singular basis shape:", basis.shape)

In [ ]:
expected_spectrum = np.array([-1.0, 1.0j, -1.0j], dtype=np.complex128)
estimated_spectrum = np.linalg.eigvals(fitted_active)
spectral_match = match_eigenvalues(estimated_spectrum, expected_spectrum)
one_step_error = np.linalg.norm(current @ fitted_operator.T - future) / np.linalg.norm(future)
projector = basis @ basis.T
invariance_error = np.linalg.norm((np.eye(4) - projector) @ fitted_operator @ basis) / np.linalg.norm(fitted_operator @ basis)
intertwining_error = np.linalg.norm(fitted_active - true_active) / np.linalg.norm(true_active)
left_eigenvalues, left_vectors = left_eigendecomposition(fitted_active)
left_residual = left_vectors @ fitted_active - left_eigenvalues[:, None] * left_vectors
max_left_residual = float(np.max(np.linalg.norm(left_residual, axis=1)))

initial = centered_phase_indicators(np.array([0]), NUM_PHASES)[0]
rollout = linear_rollout(fitted_operator, initial, steps=8)
rollout_truth = centered_phase_indicators(np.arange(9) % NUM_PHASES, NUM_PHASES)
rollout_errors = np.array([
    np.linalg.norm(rollout[step] - rollout_truth[step]) / np.linalg.norm(rollout_truth[step])
    for step in range(1, 9)
])

metrics = {
    "active_rank": basis.shape[1],
    "one_step_relative_error": float(one_step_error),
    "active_invariance_error": float(invariance_error),
    "intertwining_error": float(intertwining_error),
    "spectral_mean_absolute_error": spectral_match["mean_absolute_error"],
    "spectral_max_absolute_error": spectral_match["max_absolute_error"],
    "max_left_eigenvector_residual": max_left_residual,
    "max_eight_step_rollout_error": float(rollout_errors.max()),
}
passed = bool(metrics["active_rank"] == 3 and all(value < TOLERANCE for key, value in metrics.items() if key != "active_rank"))
print(json.dumps({**metrics, "passed": passed}, indent=2))
print("Matched estimated spectrum:", spectral_match["estimated"])
print("Matched expected spectrum: ", spectral_match["expected"])

In [ ]:
mode_index = int(np.argmin(np.abs(left_eigenvalues - 1.0j)))
ambient_left_mode = left_vectors[mode_index] @ basis.T
phase_features = centered_phase_indicators(np.arange(4), NUM_PHASES)
mode_values = phase_features @ ambient_left_mode
mode_future = centered_phase_indicators((np.arange(4) + 1) % 4, NUM_PHASES) @ ambient_left_mode
mode_residual = mode_future - left_eigenvalues[mode_index] * mode_values
assert np.max(np.abs(mode_residual)) < TOLERANCE

fig, axes = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)
image = axes[0].imshow(fitted_operator, cmap="coolwarm", vmin=-1, vmax=1)
axes[0].set(title="Operador ajustado (ambiente)", xlabel="Fase actual", ylabel="Fase futura")
fig.colorbar(image, ax=axes[0], fraction=0.046)
axes[1].scatter(expected_spectrum.real, expected_spectrum.imag, s=120, marker="x", label="esperado")
axes[1].scatter(estimated_spectrum.real, estimated_spectrum.imag, s=60, facecolors="none", edgecolors="tab:orange", label="estimado")
circle = plt.Circle((0, 0), 1, fill=False, linestyle=":", color="gray")
axes[1].add_patch(circle)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].axvline(0, color="black", linewidth=0.5)
axes[1].set(title="Espectro activo", xlabel="Real", ylabel="Imaginaria", aspect="equal", xlim=(-1.2, 1.2), ylim=(-1.2, 1.2))
axes[1].legend()
axes[2].semilogy(np.arange(1, 9), np.maximum(rollout_errors, 1e-18), marker="o")
axes[2].axhline(TOLERANCE, color="tab:red", linestyle="--", label="gate")
axes[2].set(title="Error de rollout", xlabel="Horizonte k", ylabel="Error relativo")
axes[2].legend()
axes[3].plot(np.arange(4), mode_values.real, marker="o", label="Re φᵢ")
axes[3].plot(np.arange(4), mode_values.imag, marker="o", label="Im φᵢ")
axes[3].set(title="Eigenfunction λ≈i", xlabel="Fase", ylabel="Valor", xticks=np.arange(4))
axes[3].legend()
plt.show()

display(Markdown(f"""## Análisis del resultado

- Gate de Etapa 1: **{'PASS' if passed else 'FAIL'}**.
- El span activo tiene dimensión **{metrics['active_rank']}**: el modo constante fue eliminado.
- Error espectral medio/máximo: **{metrics['spectral_mean_absolute_error']:.2e} / {metrics['spectral_max_absolute_error']:.2e}**.
- Error de entrelazamiento activo: **{metrics['intertwining_error']:.2e}**.
- Máximo error de rollout hasta ocho pasos: **{metrics['max_eight_step_rollout_error']:.2e}**.
- Residual máximo de left eigenvectors: **{metrics['max_left_eigenvector_residual']:.2e}**.

La matriz ambiente de mínimos cuadrados puede actuar arbitrariamente sobre la dirección constante que los datos no usan. El espectro relevante es el de su restricción al span activo, donde recuperamos `−1`, `i` y `−i`. Esta validación evita confundir un eigenvalue extra no identificado con un modo Koopman aprendido.
"""))